# Week 16 — Thursday (Jul 9): NeRF & 3D Gaussian Splatting

## Learning Objectives
- Understand Neural Radiance Fields (NeRF) — how a scene is represented as an MLP
- Derive the volume rendering equation used to render images from the NeRF
- Know why positional encoding is essential for NeRF to learn high-frequency details
- Implement a simplified 2D NeRF-style model (Instant-NeRF style concept)
- Understand 3D Gaussian Splatting (3DGS) and why it is 100× faster than NeRF

## Key Concepts
1. **NeRF** — scene = continuous function `(x,y,z,θ,φ) → (RGB, density σ)` represented by an MLP
2. **Volume rendering** — integrate colour + density along a camera ray to produce a pixel colour
3. **Positional encoding** — map coordinates to high-frequency Fourier features before MLP input
4. **3DGS** — scene = millions of 3D Gaussians with colour/opacity; renders by projecting + splatting to screen
5. **NeRF vs 3DGS** — NeRF: slow train, slow render, smooth; 3DGS: fast both, real-time 30+ fps

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F

plt.rcParams['figure.figsize'] = (14, 6)
np.random.seed(42)
torch.manual_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | Device: {DEVICE}')
print('Week 16 — NeRF & 3D Gaussian Splatting')

## Section 1: What is NeRF?

Mildenhall et al. (2020, ECCV) — "Representing Scenes as Neural Radiance Fields for View Synthesis"

### Core Idea
Instead of storing a 3D scene as a mesh or voxel grid, NeRF stores it as the **weights of an MLP**:

```
INPUT:   (x, y, z)        — 3D point in space
         (θ, φ)           — viewing direction (azimuth, elevation)
         ↓
      MLP (8 FC layers)
         ↓
OUTPUT:  (R, G, B)        — colour at this point from this direction
         σ                — density (how opaque this point is)
```

### Volume Rendering Equation
To get the pixel colour for a camera ray `r(t) = o + t·d`:

$$C(r) = \int_{t_n}^{t_f} T(t) \cdot \sigma(r(t)) \cdot c(r(t), d) \; dt$$

where transmittance $T(t) = \exp\left(-\int_{t_n}^{t} \sigma(r(s)) ds\right)$

**Discrete approximation** (N sample points along ray):
$$\hat{C}(r) = \sum_{i=1}^{N} T_i (1 - e^{-\sigma_i \delta_i}) c_i$$
$$T_i = \exp\left(-\sum_{j=1}^{i-1} \sigma_j \delta_j \right)$$

- $\delta_i$ = distance between sample $i$ and $i+1$
- $(1 - e^{-\sigma_i \delta_i})$ = alpha (opacity) of sample $i$
- $T_i$ = how much light has passed through before reaching sample $i$

In [ ]:
# ── Visualise volume rendering along a ray ─────────────────────────────────────
t_vals = np.linspace(0, 1, 64)

# Simulate density: two objects along the ray
sigma = (np.exp(-((t_vals - 0.3)**2) / 0.005) * 30 +
         np.exp(-((t_vals - 0.7)**2) / 0.003) * 20)

# Simulate colour: object 1 = red, object 2 = blue
colour = np.zeros((64, 3))
colour[:, 0] = np.exp(-((t_vals - 0.3)**2) / 0.01)  # red
colour[:, 2] = np.exp(-((t_vals - 0.7)**2) / 0.008) # blue

# Compute transmittance T
delta = t_vals[1] - t_vals[0]
alpha = 1 - np.exp(-sigma * delta)
T     = np.cumprod(np.concatenate([[1], 1 - alpha[:-1]]))
weights = T * alpha
final_colour = (weights[:, None] * colour).sum(axis=0)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].plot(t_vals, sigma, color='#e74c3c', lw=2)
axes[0].fill_between(t_vals, sigma, alpha=0.2, color='#e74c3c')
axes[0].set_title('Density σ(t) along ray', fontweight='bold'); axes[0].set_xlabel('t')
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_vals, T, color='#3498db', lw=2)
axes[1].set_title('Transmittance T(t)\n(how much light passes through)', fontweight='bold')
axes[1].set_xlabel('t'); axes[1].grid(True, alpha=0.3)

axes[2].plot(t_vals, weights, color='#2ecc71', lw=2)
axes[2].fill_between(t_vals, weights, alpha=0.2, color='#2ecc71')
axes[2].set_title('Weights T·α\n(contribution of each sample)', fontweight='bold')
axes[2].set_xlabel('t'); axes[2].grid(True, alpha=0.3)

# Final colour swatch
axes[3].add_patch(plt.Rectangle((0,0),1,1, color=np.clip(final_colour,0,1)))
axes[3].set_xlim(0,1); axes[3].set_ylim(0,1)
axes[3].set_title(f'Rendered pixel colour\nR={final_colour[0]:.2f} G={final_colour[1]:.2f} B={final_colour[2]:.2f}',
                  fontweight='bold')
axes[3].axis('off')

plt.suptitle('NeRF Volume Rendering — Integrating Along a Camera Ray', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('nerf_volume_rendering.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved nerf_volume_rendering.png')

## Section 2: Positional Encoding — Why MLPs Need Fourier Features

An MLP with raw (x, y, z) coordinates as input produces very smooth, blurry outputs — it cannot learn high-frequency details like sharp edges and fine textures.

**NeRF fix**: map coordinates to high-frequency Fourier features before feeding to MLP:

$$\gamma(p) = \left( \sin(2^0 \pi p),\; \cos(2^0 \pi p),\; \sin(2^1 \pi p),\; \cos(2^1 \pi p),\; \ldots,\; \sin(2^{L-1} \pi p),\; \cos(2^{L-1} \pi p) \right)$$

- For each coordinate we produce $2L$ values
- NeRF paper uses $L=10$ for position (xyz) and $L=4$ for direction (θ,φ)
- Total input size: $3 \times 2 \times 10 + 2 \times 2 \times 4 = 60 + 16 = 76$ (plus original 5 = 81)

In [ ]:
def positional_encoding(x, L=10):
    """
    Map scalar x to 2L Fourier features.
    x: (N,) or (N, D)  →  output: (N, 2*L) or (N, D*2*L)
    """
    freqs = 2.0 ** torch.arange(L, dtype=torch.float32) * torch.pi
    if x.dim() == 1:
        x = x.unsqueeze(-1)  # (N,1)
    xf = x.unsqueeze(-1) * freqs  # (N, D, L)
    enc = torch.cat([torch.sin(xf), torch.cos(xf)], dim=-1)  # (N, D, 2L)
    return enc.flatten(1)  # (N, D*2L)


# Demonstrate: fitting a 1D signal with vs without positional encoding
x = torch.linspace(-1, 1, 200)
# Target: high-frequency signal
y_target = torch.sin(10 * x) * torch.cos(5 * x) + 0.5 * torch.sin(25 * x)

def train_mlp(x_in, y_tgt, hidden=64, epochs=500):
    net = nn.Sequential(
        nn.Linear(x_in.shape[1], hidden), nn.ReLU(),
        nn.Linear(hidden, hidden),        nn.ReLU(),
        nn.Linear(hidden, 1)
    )
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    for _ in range(epochs):
        loss = F.mse_loss(net(x_in).squeeze(), y_tgt)
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        return net(x_in).squeeze().numpy()

x_raw = x.unsqueeze(1)                   # (200, 1)  — raw coordinate
x_enc = positional_encoding(x, L=8)      # (200, 16) — Fourier features

print('Training MLP without positional encoding...')
y_raw = train_mlp(x_raw, y_target)
print('Training MLP WITH positional encoding...')
y_enc = train_mlp(x_enc, y_target)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(x.numpy(), y_target.numpy(), color='#2ecc71', lw=2, label='Target')
axes[0].set_title('Target Signal (high-frequency)', fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(x.numpy(), y_target.numpy(), color='#2ecc71', lw=2, alpha=0.4, label='Target')
axes[1].plot(x.numpy(), y_raw, color='#e74c3c', lw=2, label='MLP (raw x)')
axes[1].set_title('Without Positional Encoding\n(MLP learns smooth, misses high-freq)', fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(x.numpy(), y_target.numpy(), color='#2ecc71', lw=2, alpha=0.4, label='Target')
axes[2].plot(x.numpy(), y_enc, color='#3498db', lw=2, label='MLP (Fourier enc)')
axes[2].set_title('With Positional Encoding L=8\n(MLP captures high-frequency details)', fontweight='bold')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.suptitle('Positional Encoding: Why NeRF Cannot Use Raw Coordinates', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('positional_encoding.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved positional_encoding.png')

## Section 3: Tiny NeRF — Fitting a 2D Image

Full 3D NeRF training takes hours. Here we implement a **2D NeRF analogue**:
- Scene is a 2D image instead of a 3D volume
- MLP maps (x, y) → (R, G, B)
- Shows the full pipeline: positional encoding → MLP → pixel colour
- Demonstrates how NeRF "memorises" a scene in its weights

In [ ]:
# ── Tiny 2D NeRF: fit an image with a coordinate MLP ──────────────────────────
import cv2

# Create a synthetic target image (geometric patterns — easy to see if NeRF fits it)
H, W = 64, 64
target = np.zeros((H, W, 3), dtype=np.float32)
# Background gradient
for i in range(H):
    for j in range(W):
        target[i,j] = [i/H, j/W, 0.5]
# Red circle
cv2.circle(target, (20,20), 12, (0.9, 0.1, 0.1), -1)
# Blue rectangle
cv2.rectangle(target, (38,38), (60,60), (0.1, 0.1, 0.9), -1)
# Green triangle
pts = np.array([[32,10],[48,10],[40,24]], dtype=np.int32)
cv2.fillPoly(target, [pts], (0.1, 0.8, 0.1))
target = np.clip(target, 0, 1)

# Build (x, y) coordinate grid
ys = torch.linspace(-1, 1, H)
xs = torch.linspace(-1, 1, W)
grid_y, grid_x = torch.meshgrid(ys, xs, indexing='ij')
coords = torch.stack([grid_x, grid_y], dim=-1).reshape(-1, 2)  # (H*W, 2)
rgb_gt = torch.from_numpy(target.reshape(-1, 3)).float()         # (H*W, 3)

# Positional encoding for 2D coords
def pos_enc_2d(coords, L=8):
    freqs = (2.0 ** torch.arange(L).float() * torch.pi)  # (L,)
    enc   = []
    for i in range(coords.shape[1]):
        c = coords[:, i:i+1]  # (N,1)
        enc += [torch.sin(c * freqs), torch.cos(c * freqs)]
    return torch.cat(enc, dim=1)  # (N, 4L)

coords_enc = pos_enc_2d(coords, L=6)  # (H*W, 24)

# Tiny NeRF MLP
class TinyNeRF(nn.Module):
    def __init__(self, in_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 3),      nn.Sigmoid()  # RGB in [0,1]
        )
    def forward(self, x): return self.net(x)

nerf = TinyNeRF(in_dim=coords_enc.shape[1]).to(DEVICE)
opt  = torch.optim.Adam(nerf.parameters(), lr=5e-4)

coords_enc_d = coords_enc.to(DEVICE)
rgb_gt_d     = rgb_gt.to(DEVICE)

# Train
N_STEPS = 1500
losses  = []
snapshots = {}

for step in range(1, N_STEPS + 1):
    pred = nerf(coords_enc_d)
    loss = F.mse_loss(pred, rgb_gt_d)
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())
    if step in [50, 300, 800, 1500]:
        with torch.no_grad():
            snapshots[step] = nerf(coords_enc_d).cpu().numpy().reshape(H, W, 3)

print(f'Final MSE: {losses[-1]:.6f}  PSNR: {-10*np.log10(losses[-1]):.2f} dB')

In [ ]:
# ── Visualise NeRF fitting progress ───────────────────────────────────────────
fig, axes = plt.subplots(1, 6, figsize=(20, 4))
axes[0].imshow(target); axes[0].set_title('Target Image', fontweight='bold'); axes[0].axis('off')

for ax, (step, img) in zip(axes[1:], snapshots.items()):
    psnr = -10 * np.log10(np.mean((img - target)**2) + 1e-8)
    ax.imshow(np.clip(img, 0, 1))
    ax.set_title(f'Step {step}\nPSNR={psnr:.1f}dB', fontweight='bold', fontsize=9)
    ax.axis('off')

axes[5].plot(range(1, N_STEPS+1), losses, color='#e74c3c', lw=1.5)
axes[5].set_title('MSE Loss\nvs Training Step', fontweight='bold', fontsize=9)
axes[5].set_yscale('log'); axes[5].grid(True, alpha=0.3)
axes[5].axis('on')

plt.suptitle('Tiny 2D NeRF — MLP Memorising a Scene in Its Weights\n'
             '(x,y) + Positional Encoding → MLP → RGB', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('tiny_nerf_fitting.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved tiny_nerf_fitting.png')

## Section 4: 3D Gaussian Splatting (3DGS)

Kerbl et al. (SIGGRAPH 2023) — completely different representation from NeRF.

### Core Idea
Represent the scene as **millions of 3D Gaussians**, each with:
- **Position** (x, y, z) — where it is in 3D space
- **Covariance** (3×3 matrix Σ) — its shape and orientation (an ellipsoid)
- **Colour** (RGB or SH coefficients) — what colour it is from each viewing angle
- **Opacity** α — how transparent it is

```
3D Gaussian:  G(x) = exp(-½ (x-μ)ᵀ Σ⁻¹ (x-μ))

To render a pixel:
1. Project all 3D Gaussians onto the 2D screen (covariance projects too)
2. Sort by depth (back to front)
3. Alpha-composite them front-to-back
   C = Σᵢ cᵢ αᵢ ∏ⱼ<ᵢ (1 - αⱼ)   ← just like volume rendering but discrete
```

### NeRF vs 3DGS
| | NeRF | 3DGS |
|--|------|------|
| Representation | MLP weights | 3D Gaussian primitives |
| Training time | Hours (GPU) | 30-60 min |
| Render speed | Seconds/frame | Real-time (30+ fps) |
| Memory | Low (MLP weights) | High (millions of Gaussians) |
| Quality | Smooth, no artifacts | Sharper, but floaters possible |
| Editing | Hard | Easier (remove Gaussians) |
| Init required | Random | SfM point cloud (COLMAP) |

In [ ]:
# ── Visualise 2D Gaussian Splatting concept ────────────────────────────────────
def gaussian_2d(x, y, cx, cy, sx, sy, angle, amplitude):
    """2D Gaussian with rotation — simulates one projected 3D Gaussian."""
    cos_a, sin_a = np.cos(angle), np.sin(angle)
    xr = cos_a*(x-cx) + sin_a*(y-cy)
    yr = -sin_a*(x-cx) + cos_a*(y-cy)
    return amplitude * np.exp(-(xr**2/(2*sx**2) + yr**2/(2*sy**2)))

H_s, W_s = 128, 128
yy, xx = np.mgrid[0:H_s, 0:W_s]

# Define some Gaussians (simulating a simple scene)
gaussians = [
    # cx, cy, sx, sy, angle, colour, alpha
    (40,  40,  15, 8,  0.5,  np.array([0.9,0.2,0.2]), 0.8),   # red ellipse
    (90,  50,  10, 10, 0.0,  np.array([0.2,0.3,0.9]), 0.8),   # blue circle
    (60,  90,  20, 6,  1.2,  np.array([0.2,0.8,0.2]), 0.75),  # green wide
    (30,  90,  8,  12, 0.8,  np.array([0.9,0.7,0.1]), 0.7),   # yellow
    (100, 100, 15, 5,  0.3,  np.array([0.7,0.2,0.8]), 0.75),  # purple
    (70,  40,  6,  6,  0.0,  np.array([0.9,0.5,0.1]), 0.8),   # orange
]

# Render: alpha compositing
canvas = np.zeros((H_s, W_s, 3))
alpha_acc = np.zeros((H_s, W_s))

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
individual = np.zeros((H_s, W_s, 3))

for k, (cx,cy,sx,sy,ang,col,alpha) in enumerate(gaussians):
    g = gaussian_2d(xx, yy, cx, cy, sx, sy, ang, 1.0)
    g_alpha = g * alpha
    # Alpha composite: C = C_prev + c_new * α_new * (1 - acc_alpha)
    transmittance = 1 - alpha_acc
    canvas += col[None,None,:] * g_alpha[:,:,None] * transmittance[:,:,None]
    alpha_acc = np.clip(alpha_acc + g_alpha * transmittance, 0, 1)
    individual += col[None,None,:] * g[:,:,None] * 0.6

axes[0].imshow(np.clip(individual,0,1))
axes[0].set_title('Individual Gaussians\n(before compositing)', fontweight='bold'); axes[0].axis('off')

axes[1].imshow(np.clip(canvas,0,1))
axes[1].set_title('Splatted Result\n(alpha composited)', fontweight='bold'); axes[1].axis('off')

# Show individual Gaussian shapes
for k, (cx,cy,sx,sy,ang,col,alpha) in enumerate(gaussians[:4]):
    g = gaussian_2d(xx, yy, cx, cy, sx, sy, ang, 1.0)
    axes[2].contour(xx, yy, g, levels=[0.1, 0.4, 0.7],
                   colors=[col.tolist()], alpha=0.8)
axes[2].set_xlim(0,W_s); axes[2].set_ylim(H_s,0)
axes[2].set_title('Gaussian Ellipses\n(3D shape projected to 2D screen)', fontweight='bold')
axes[2].set_aspect('equal'); axes[2].grid(True, alpha=0.2)

# NeRF vs 3DGS comparison
axes[3].axis('off')
txt = ('Rendering Pipeline Comparison\n'
       '══════════════════════════════════\n\n'
       'NeRF:\n'
       '  For each pixel:\n'
       '  1. Cast ray through pixel\n'
       '  2. Sample 64-192 points\n'
       '  3. Run MLP at each point\n'
       '  4. Volume integrate\n'
       '  ❌ Very slow (seconds/frame)\n\n'
       '3DGS:\n'
       '  For each frame:\n'
       '  1. Project Gaussians → 2D\n'
       '  2. Sort by depth (GPU)\n'
       '  3. Rasterise + composite\n'
       '  ✅ Real-time (30+ fps)')
axes[3].text(0.05, 0.95, txt, va='top', fontfamily='monospace',
             fontsize=8.5, transform=axes[3].transAxes,
             bbox=dict(boxstyle='round', facecolor='#f0f9ff', alpha=0.9))
axes[3].set_title('NeRF vs 3DGS Pipeline', fontweight='bold')

plt.suptitle('3D Gaussian Splatting — Scene as 3D Ellipsoids Projected to Screen',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('gaussian_splatting.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved gaussian_splatting.png')

In [ ]:
# ── Summary figure ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# NeRF training overview
nerf_txt = (
    'NeRF PIPELINE (Mildenhall 2020)\n'
    '═══════════════════════════════════════\n\n'
    'INPUT  : 20-100 posed photos of scene\n'
    'OUTPUT : novel view synthesis\n\n'
    'Step 1: Camera rays from each pixel\n'
    '        r(t) = o + t·d\n\n'
    'Step 2: Sample N=64 points per ray\n'
    '        Coarse + Fine sampling\n\n'
    'Step 3: Positional encoding\n'
    '        (x,y,z,θ,φ) → 81-dim Fourier\n\n'
    'Step 4: MLP forward pass (8 layers)\n'
    '        → σ (density) + RGB\n\n'
    'Step 5: Volume rendering equation\n'
    '        C = Σ Tᵢ αᵢ cᵢ\n\n'
    'Loss  : MSE(rendered, real pixel)\n'
    'Train : ~100K iters, hours on GPU\n'
    'Render: ~30 sec per frame\n'
)
axes[0].text(0.03, 0.97, nerf_txt, va='top', fontfamily='monospace',
             fontsize=9, transform=axes[0].transAxes,
             bbox=dict(boxstyle='round', facecolor='#fff3e0', alpha=0.9))
axes[0].set_title('NeRF — Scene as MLP Weights', fontweight='bold'); axes[0].axis('off')

# 3DGS overview
gs_txt = (
    '3DGS PIPELINE (Kerbl 2023)\n'
    '═══════════════════════════════════════\n\n'
    'INPUT  : SfM point cloud + photos\n'
    'OUTPUT : real-time novel view synthesis\n\n'
    'Step 1: Initialise Gaussians from SfM\n'
    '        (COLMAP sparse reconstruction)\n\n'
    'Step 2: Optimise per-Gaussian params\n'
    '        μ (pos), Σ (shape), c, α\n\n'
    'Step 3: Adaptive density control\n'
    '        Clone small Gaussians\n'
    '        Split large Gaussians\n\n'
    'Step 4: Rasterise with custom CUDA\n'
    '        Project → sort → composite\n\n'
    'Loss  : L1 + SSIM (image quality)\n'
    'Train : 30-60 min on GPU\n'
    'Render: 30-100 fps real-time\n'
)
axes[1].text(0.03, 0.97, gs_txt, va='top', fontfamily='monospace',
             fontsize=9, transform=axes[1].transAxes,
             bbox=dict(boxstyle='round', facecolor='#e8f5e9', alpha=0.9))
axes[1].set_title('3DGS — Scene as 3D Gaussian Primitives', fontweight='bold'); axes[1].axis('off')

plt.suptitle('Week 16 Thu — NeRF vs 3D Gaussian Splatting: Full Pipeline Summary',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('nerf_vs_3dgs_summary.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved nerf_vs_3dgs_summary.png')

## Key Takeaways

| Concept | Summary |
|---------|--------|
| **NeRF** | Scene stored as MLP weights — `(x,y,z,θ,φ) → (RGB, σ)` |
| **Volume rendering** | Integrate colour × density along ray: `C = Σ Tᵢ αᵢ cᵢ` |
| **Transmittance** | How much light passes through before point i — decreases as density accumulates |
| **Positional encoding** | Fourier features let MLP learn high-freq details — without it output is blurry |
| **3DGS** | Scene stored as 3D Gaussians — each has position, shape (Σ), colour, opacity |
| **Splatting** | Project 3D Gaussians to 2D screen → sort by depth → alpha composite |
| **3DGS speed** | Real-time rendering because rasterisation is fast on GPU — no ray marching |
| **Instant-NGP** | NeRF variant using hash encoding instead of Fourier — trains in minutes |

### Tools to Try
1. **nerfstudio** — `pip install nerfstudio` → `ns-train nerfacto --data your_images/` — easiest NeRF pipeline
2. **gaussian-splatting** — original CUDA implementation from Kerbl et al. (GitHub: graphdeco-inria)
3. **gsplat** — PyTorch bindings for 3DGS, easier to experiment with
4. **NeRFStudio viewer** — interactive 3D viewer in your browser while training

### Exercises
1. Increase the positional encoding L from 4 to 16 in the 2D NeRF — how much sharper does it get?
2. Train the Tiny NeRF for 5000 steps — what PSNR do you achieve?
3. Add a second MLP head for density σ in the 2D demo and implement real volume rendering
4. Install nerfstudio and run it on a video you take of an object (phone camera, 360° rotation)
5. Compare PSNR/SSIM of NeRF vs 3DGS on the same scene using nerfstudio benchmarks